# Phase 7 (Prior Case Retrieval) — c-6 knowledge distillation, LORMA teacher

Teacher = your Phase 7 c-6 LORMA retriever ((I + BA)W, loaded from the LORMA notebook's output, frozen). Student = a shallower Phase 1 c-6 encoder (KEEP_ENC_LAYERS blocks, no decoder), trained on gold labels plus the teacher's similarity scores (listwise KL or Margin-MSE).

Attach as inputs: the Phase 1 c-6 notebook output (Cell A ROOT) and the Phase 7 c-6 LORMA notebook output (`c6_lorma_pcr_*/teacher_config.json` + `best_trainable.pt`). **Fill in only section [H] of Cell K2.**

## Cell 0 — Phase 1 imports, HF login, tokenizer class, model classes

In [1]:
# --- Phase 1 c-6 notebook, cell 1 (verbatim) ---
from collections import Counter
from tqdm.auto import tqdm
import pickle
import re
from collections import Counter, defaultdict


# HF login (Phase 1 cell 2), needed if the dataset download asks for auth
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
# --- Phase 1 c-6 notebook, cell 8 (verbatim): SPECIAL_TOKENS, BPETokenizer, train_tokenizer ---


SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]  # fixed ids 0-3, never merged

class BPETokenizer:
    def __init__(self):
        self.id_to_bytes = {}
        self.bytes_to_id = {}
        self.merges = {}
        self.merge_order = []
        self.merge_ranks = {}  # Precomputed pair -> rank for fast encoding
        self.cache = {}        # Chunk lookup cache
        self._vocab_size = 0
        
        # Splits on whitespace+word or just whitespace. 
        # Guarantees round-trip: "".join(chunks) == text
        self.split_pattern = re.compile(r'\s*\S+|\s+')

    def train(self, text: str, vocab_size: int):
        # 1. Setup base vocabulary
        for i, tok in enumerate(SPECIAL_TOKENS):
            b = tok.encode("utf-8")
            self.id_to_bytes[i] = b
            self.bytes_to_id[b] = i

        next_id = len(SPECIAL_TOKENS)
        byte_to_base_id = {}
        for value in range(256):
            b = bytes([value])
            self.id_to_bytes[next_id] = b
            self.bytes_to_id[b] = next_id
            byte_to_base_id[value] = next_id
            next_id += 1

        # 2. Split text into chunks and get frequencies
        chunks = self.split_pattern.findall(text)
        chunk_counts = Counter(chunks)
        
        # 3. Store each distinct chunk as a tuple of base byte ids
        chunk_to_ids = {
            chunk: tuple(byte_to_base_id[b] for b in chunk.encode("utf-8"))
            for chunk in chunk_counts
        }

        # 4. Initialize weighted pair counts and the pair-to-chunk index
        pair_counts = Counter()
        pair_to_chunks = defaultdict(set)
        
        def get_pairs(ids):
            return [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]

        for chunk, ids in chunk_to_ids.items():
            count = chunk_counts[chunk]
            for pair in get_pairs(ids):
                pair_counts[pair] += count
                pair_to_chunks[pair].add(chunk)

        target = vocab_size
        num_merges = target - next_id
        pbar = tqdm(total=num_merges, desc="Training BPE")

        # 5. Iteratively merge the best pair
        while next_id < target:
            if not pair_counts:
                break

            max_freq = max(pair_counts.values())
            if max_freq <= 1:
                break

            # Min tuple tie-breaker matching your original behavior
            best_pair = min(p for p, c in pair_counts.items() if c == max_freq)

            new_id = next_id
            self.merges[best_pair] = new_id
            self.merge_order.append(best_pair)
            self.merge_ranks[best_pair] = len(self.merge_order) - 1
            
            self.id_to_bytes[new_id] = self.id_to_bytes[best_pair[0]] + self.id_to_bytes[best_pair[1]]
            self.bytes_to_id[self.id_to_bytes[new_id]] = new_id

            # 6. Update only the chunks containing the merged pair
            chunks_to_update = list(pair_to_chunks[best_pair])
            for chunk in chunks_to_update:
                old_ids = chunk_to_ids[chunk]
                count = chunk_counts[chunk]
                
                # Subtract old pair combinations
                for pair in get_pairs(old_ids):
                    pair_counts[pair] -= count
                    if pair_counts[pair] <= 0:
                        del pair_counts[pair]
                    if chunk in pair_to_chunks[pair]:
                        pair_to_chunks[pair].remove(chunk)
                
                # Apply the merge inside this chunk
                new_ids = []
                i = 0
                while i < len(old_ids):
                    if i < len(old_ids) - 1 and (old_ids[i], old_ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(old_ids[i])
                        i += 1
                new_ids = tuple(new_ids)
                chunk_to_ids[chunk] = new_ids
                
                # Add new pair combinations created by the merge
                for pair in get_pairs(new_ids):
                    pair_counts[pair] += count
                    pair_to_chunks[pair].add(chunk)

            next_id += 1
            pbar.update(1)

        pbar.close()
        self._vocab_size = next_id

    def encode(self, text: str):
        chunks = self.split_pattern.findall(text)
        out_ids = []
        
        for chunk in chunks:
            # Check the dictionary cache first
            if chunk in self.cache:
                out_ids.extend(self.cache[chunk])
                continue
            
            # Cache miss: break into bytes and process
            chunk_bytes = chunk.encode("utf-8")
            ids = [self.bytes_to_id[bytes([b])] for b in chunk_bytes]
            
            # Merge until no adjacent pair has a valid merge rank
            while len(ids) > 1:
                pairs = [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]
                best_pair = None
                best_rank = float('inf')
                
                # Find the pair with the lowest rank
                for p in pairs:
                    rank = self.merge_ranks.get(p, float('inf'))
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = p
                
                if best_pair is None:
                    break  # No more valid merges for this chunk
                    
                # Apply the best merge to all occurrences in the chunk
                new_id = self.merges[best_pair]
                new_ids = []
                i = 0
                while i < len(ids):
                    if i < len(ids) - 1 and (ids[i], ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(ids[i])
                        i += 1
                ids = new_ids
            
            # Store result in cache and append
            self.cache[chunk] = ids
            out_ids.extend(ids)
            
        return out_ids

    def decode(self, ids):
        out = b"".join(self.id_to_bytes[i] for i in ids)
        return out.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self._vocab_size

    @property
    def vocab_size(self):
        return self._vocab_size


def train_tokenizer(train_text: str, vocab_size: int):
    """Train the A1 BPE tokenizer on `train_text` and return it, ready for A3."""
    tok = BPETokenizer()
    tok.train(train_text, vocab_size)
    return tok


In [3]:
# --- Phase 1 c-6 notebook, cells 20, 22 and 23 (verbatim, in this order) ---
# Cell 23 must come after cell 22: it redefines CausalSelfAttention (q/k/v_proj), which the
# saved decoder weights use. Without it, Cell A fails with qkv_proj key errors in decoder_layers.

# ---- cell 20 ----
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---- cell 22 ----
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Seq2SeqConfig:
    vocab_size: int
    d_model: int = 512
    n_heads: int = 8
    d_ff: int = 2048
    max_seq_len: int = 512
    dropout: float = 0.1
    n_layers: int = 8

# ---------------------------------------------------------
# 1. Attention Modules
# ---------------------------------------------------------
class EncoderSelfAttention(nn.Module):
    """Bidirectional self-attention for the encoder (no causal mask)."""
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv_proj(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in qkv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class CausalSelfAttention(nn.Module):
    """Causal self-attention for the decoder (looks only at past tokens)."""
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("mask", mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv_proj(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in qkv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.mask[:T, :T], float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class CrossAttention(nn.Module):
    """Cross-attention: Query from Decoder, Key/Value from Encoder."""
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.kv_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        B_c, T_c, C_c = context.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        kv = self.kv_proj(context).chunk(2, dim=-1)
        k, v = [t.view(B_c, T_c, self.n_heads, self.d_head).transpose(1, 2) for t in kv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

# ---------------------------------------------------------
# 2. Feed-Forward & Standard Blocks
# ---------------------------------------------------------
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor):
        return self.net(x)


class EncoderBlock(nn.Module):
    """Standard bidirectional Transformer encoder block."""
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = EncoderSelfAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.self_attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class DecoderBlock(nn.Module):
    """Standard causal Transformer decoder block with cross-attention."""
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout)
        
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.cross_attn = CrossAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        
        self.ln3 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor, encoder_out: torch.Tensor) -> torch.Tensor:
        x = x + self.self_attn(self.ln1(x))
        x = x + self.cross_attn(self.ln2(x), encoder_out)
        x = x + self.ffn(self.ln3(x))
        return x


# ---------------------------------------------------------
# 4. Hourglass Blocks (Encoder & Decoder versions)
# ---------------------------------------------------------
class EncoderHourglassBlock(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig, factor: int, bottleneck_layers: int):
        super().__init__()
        self.factor = factor
        self.downsample = Downsample(factor)
        self.bottleneck = nn.ModuleList([EncoderBlock(cfg) for _ in range(bottleneck_layers)])
        self.upsample = Upsample(factor)

    def forward(self, x: torch.Tensor):
        residual = x
        shortened_x = self.downsample(x) # No shift for Encoder
        for block in self.bottleneck:
            shortened_x = block(shortened_x)
        expanded_x = self.upsample(shortened_x)
        return expanded_x + residual


class DecoderHourglassBlock(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig, factor: int, bottleneck_layers: int):
        super().__init__()
        self.factor = factor
        self.downsample = Downsample(factor)
        self.bottleneck = nn.ModuleList([DecoderBlock(cfg) for _ in range(bottleneck_layers)])
        self.upsample = Upsample(factor)

    def forward(self, x: torch.Tensor, encoder_out: torch.Tensor):
        residual = x
        x_shifted = shift_right(x, shift=self.factor - 1) # Shift required for Decoder causality
        shortened_x = self.downsample(x_shifted)
        for block in self.bottleneck:
            shortened_x = block(shortened_x, encoder_out)
        expanded_x = self.upsample(shortened_x)
        return expanded_x + residual

# ---------------------------------------------------------
# 5. Top-Level Hourglass Seq2Seq Model
# ---------------------------------------------------------
class StandardEncoderDecoder(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)

        self.encoder_layers = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.encoder_ln = nn.LayerNorm(cfg.d_model)

        self.decoder_layers = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.decoder_ln = nn.LayerNorm(cfg.d_model)

        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

    def get_positional_embeddings(self, seq_len, device):
        return self.pos_emb(torch.arange(seq_len, device=device).unsqueeze(0))

    def encode(self, enc_idx):
        x = self.tok_emb(enc_idx) + self.get_positional_embeddings(enc_idx.size(1), enc_idx.device)
        for block in self.encoder_layers:
            x = block(x)
        return self.encoder_ln(x)

    def decode(self, dec_idx, encoder_out):
        x = self.tok_emb(dec_idx) + self.get_positional_embeddings(dec_idx.size(1), dec_idx.device)
        for block in self.decoder_layers:
            x = block(x, encoder_out)
        return self.lm_head(self.decoder_ln(x))

    def forward(self, enc_idx, dec_idx):
        return self.decode(dec_idx, self.encode(enc_idx))


# ---- cell 23 ----
# ============================================================
# CELL 1 — Decoder-only Transformer (your A3 Q1 code, unchanged except the 2 marked LayerNorm lines)
# Replaces: §9 "Define The Model" (GPT2Config / GPT2LMHeadModel) and the DataCollator cell.
# ============================================================
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------- A3 Q1.1: manual causal multi-head self-attention ----------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head)
        q = q.transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head)
        k = k.transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head)
        v = v.transpose(1, 2)

        scores = q @ k.transpose(-2, -1) * self.attention_scale
        mask = self.causal_mask[:T, :T]
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


# ---------- A3 Q1.2: decoder block and language model ----------
@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 560
    n_layers: int = 6
    n_heads: int = 8
    n_kv_heads: int = 4          # == n_heads for plain MHA; unused in this version
    d_ff: int = 2000
    max_seq_len: int = 256
    dropout: float = 0.15
    pos_emb_type: str = "sinusoidal"
    attention_scale: float = None
    norm_first: bool = False     # True = pre-norm, False = post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    pe = torch.zeros(max_len, d_model)
    for a in range(max_len):
        for b in range(d_model):
            i = b // 2
            angle = a / (10000 ** (2 * i / d_model))
            if b % 2 == 0:
                pe[a][b] = math.sin(angle)
            else:
                pe[a][b] = math.cos(angle)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))      # FIX: your A3 code used self.ln1 here (ln2 was never used)
        else:
            x = self.ln1(x + self.attn(x))     # FIX: your A3 code used self.ln2 here (ln1 was never used)
            x = self.ln2(x + self.ffn(x))
        return x


@dataclass
class EncDecConfig(Seq2SeqConfig):
    enc_len: int = None      # tokens the encoder reads (prompt side)
    dec_len: int = None      # tokens the decoder learns to produce (continuation side)


class EncDecLM(nn.Module):
    def __init__(self, cfg: EncDecConfig):
        super().__init__()
        assert None not in (cfg.enc_len, cfg.dec_len), "set enc_len and dec_len in the config"
        self.cfg = cfg
        self.core = StandardEncoderDecoder(cfg)

    def encode(self, enc_idx):
        return self.core.encode(enc_idx)

    def decode(self, dec_idx, encoder_out):
        return self.core.decode(dec_idx, encoder_out)

    def forward(self, enc_idx, dec_idx):
        return self.core(enc_idx, dec_idx)


## Cell A — load the Phase 1 c-6 model and tokenizer

In [4]:
# =====================================================================================
# CELL A — load the Phase 1 c-6 encoder-decoder and its tokenizer
# Expect <All keys matched successfully>, 55.13M params, vocab 11022, d_model 560, 8 layers, max_seq_len 1536
# =====================================================================================
import torch, pickle, json, re, os
ROOT = "/kaggle/input/notebooks/rudraalt/phase-1-c-6-simple-encoder-decoder"   # TODO: the folder the find command showed (up to, not including, cs781_phase1_full)
device = torch.device("cuda")

tok = pickle.load(open(f"{ROOT}/cs781_phase1_full/model_encdec/bpe_tokenizer.pkl", "rb"))   # c-6's own tokenizer
ck = torch.load(f"{ROOT}/cs781_phase1_full/model_encdec/model_final.pt", map_location="cpu", weights_only=False)
cfg = EncDecConfig(**ck["config"])
lm = EncDecLM(cfg); print(lm.load_state_dict(ck["model"])); lm.to(device).eval()
del ck
print("params", sum(p.numel() for p in lm.parameters())/1e6, "M | vocab", tok.vocab_size,
      "| d_model", cfg.d_model, "| n_layers", cfg.n_layers, "| max_seq_len", cfg.max_seq_len)


<All keys matched successfully>
params 55.13008 M | vocab 11022 | d_model 560 | n_layers 8 | max_seq_len 1536


## Cell B — encoder sanity check

In [5]:
# =====================================================================================
# CELL B — per-token vectors from the c-6 ENCODER + sanity check
# The encoder is bidirectional: every token sees the tokens on both sides. The decoder is not used.
# =====================================================================================
def features(idx):                        # idx: (B, T) LongTensor, T <= 1536
    c = lm.core
    T = idx.size(1)
    assert T <= c.cfg.max_seq_len         # learned position table has 1536 rows
    x = c.tok_emb(idx) + c.get_positional_embeddings(T, idx.device)
    for blk in c.encoder_layers:          # 8 bidirectional EncoderBlocks
        x = blk(x)
    return c.encoder_ln(x)                # (B, T, 560)

# Sanity check: same as the model's own encoder
idx = torch.randint(4, cfg.vocab_size, (1, 64), device=device)
with torch.no_grad():
    f, ref = features(idx), lm.encode(idx)
print("max diff", (f - ref).abs().max().item(), tuple(f.shape))   # expect 0.0, (1, 64, 560)


max diff 0.0 (1, 64, 560)


## Cell C — Phase 7 PCR data (queries + candidate pool per split)

In [6]:
# =====================================================================================
# CELL C — [ANCILLARY] Phase 7 (Prior Case Retrieval) data: queries + candidate pools per split
#   Train/train.json + train_candidates.json, Dev/dev.json + dev_candidates.json,
#   Test/test.json + test_candidates.json. Each split's queries are ranked ONLY against
#   that split's own candidate pool. "text" = the case split into sentences (joined here).
# The exact JSON layout is not in the spec, so the reader accepts the usual layouts and
# prints what it found. Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
import json, random
import numpy as np
from huggingface_hub import hf_hub_download, list_repo_files

REPO = "Exploration-Lab/CS-781-Capstone"
P7_DIR = "Phase_7_PCR"
print("Phase 7 files on HF:", [f for f in list_repo_files(REPO, repo_type="dataset") if "Phase_7" in f])

def get7(name):
    """Reads Phase_7_PCR/<name>. Accepts plain JSON or JSON-lines."""
    p = hf_hub_download(REPO, f"{P7_DIR}/{name}", repo_type="dataset")
    with open(p, encoding="utf-8") as f:
        txt = f.read()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        return [json.loads(line) for line in txt.splitlines() if line.strip()]

ID_FIELDS   = ("id", "query_id", "candidate_id", "case_id", "qid", "cid", "doc_id")
TEXT_FIELDS = ("text", "document", "case_text", "content", "judgment", "sentences")
REL_FIELDS  = ("relevant_candidates", "relevant_cases", "relevant", "positives", "gold", "labels")
CAND_FIELDS = ("candidates", "candidate_ids", "candidate_pool")   # only if a query lists its own pool

def norm_id(x):
    """IDs as strings. Integer IDs are zero-padded to 10 digits like the spec's example output."""
    if isinstance(x, (int, np.integer)):
        return f"{int(x):010d}"
    return str(x).strip()

def join_text(t):
    if isinstance(t, dict):                              # e.g. {"sentences": [...]}
        t = next((t[f] for f in TEXT_FIELDS if f in t), list(t.values()))
    if isinstance(t, (list, tuple)):
        return " ".join(str(s).strip() for s in t if str(s).strip())
    return str(t)

def _unwrap(raw):
    # {"data": [...]} / {"queries": {...}} -> inner; never unwrap {"<case id>": {...}}
    while (isinstance(raw, dict) and len(raw) == 1 and isinstance(next(iter(raw.values())), (list, dict))
           and not norm_id(next(iter(raw))).isdigit()):
        raw = next(iter(raw.values()))
    idf = next((f for f in ID_FIELDS if isinstance(raw, dict) and isinstance(raw.get(f), list)), None)
    if idf:                                              # columns layout {"id": [...], "text": [...], ...}
        keys = list(raw)
        raw = [{k: raw[k][i] for k in keys} for i in range(len(raw[idf]))]
    return raw

def as_cases(raw, name, is_query):
    """-> {case_id: {"text": str, "rel": [ids] or None, "cands": [ids] or None}} in file order."""
    raw = _unwrap(raw)
    items = list(raw.items()) if isinstance(raw, dict) else [(None, d) for d in raw]
    out, n_dup = {}, 0
    for k, d in items:
        if not isinstance(d, dict):                      # {id: "text"} or {id: [sentences]}
            d = {"text": d}
        idf = next((f for f in ID_FIELDS if f in d), None)
        cid = norm_id(d[idf] if idf else k)
        assert cid not in (None, "None"), f"{name}: record without an id (keys {list(d)})"
        tf = next((f for f in TEXT_FIELDS if f in d), None)
        assert tf is not None, f"{name} {cid}: no text field (keys {list(d)})"
        rf = next((f for f in REL_FIELDS if f in d), None) if is_query else None
        cf = next((f for f in CAND_FIELDS if f in d), None) if is_query else None
        rec = {"text": join_text(d[tf]),
               "rel": [norm_id(x) for x in d[rf]] if rf and d[rf] is not None else None,
               "cands": [norm_id(x) for x in d[cf]] if cf and d[cf] is not None else None}
        if cid in out:
            n_dup += 1
            if out[cid]["text"] != rec["text"]:
                print(f"WARNING {name}: id {cid} repeated with different text, keeping the first")
            continue
        out[cid] = rec
    print(f"{name}: {len(items)} records -> {len(out)} cases (repeated ids skipped {n_dup})")
    return out

def _peek(raw, name):
    r = _unwrap(raw)
    first = r[0] if isinstance(r, list) else next(iter(r.values()))
    print(f"{name} layout: {type(r).__name__} of {len(r)} | first record:",
          list(first) if isinstance(first, dict) else type(first).__name__)

queries, pools = {}, {}
for split, folder in (("train", "Train"), ("dev", "Dev"), ("test", "Test")):
    _q = get7(f"{folder}/{split}.json"); _peek(_q, f"{split}.json")
    _c = get7(f"{folder}/{split}_candidates.json"); _peek(_c, f"{split}_candidates.json")
    queries[split] = as_cases(_q, f"{split} queries", True)
    pools[split] = as_cases(_c, f"{split} candidates", False)
del _q, _c

# test labels are private: ignore anything that looks like them
for d in queries["test"].values():
    d["rel"] = None

# Gold sets (train/dev): drop relevant ids that are not in the split's pool (cannot be retrieved)
GOLD = {}
for split in ("train", "dev"):
    pool, g, n_missing, n_rel = pools[split], {}, 0, 0
    for q, d in queries[split].items():
        rel = list(dict.fromkeys(d["rel"] or []))
        n_rel += len(rel)
        keep = [c for c in rel if c in pool and c != q]
        n_missing += len(rel) - len(keep)
        g[q] = set(keep)
    GOLD[split] = g
    n_empty = sum(not s for s in g.values())
    print(f"{split}: {n_rel} relevant links, {n_missing} not in the {split} pool (dropped) | "
          f"queries with no retrievable relevant case: {n_empty}")

# Queries with no gold cannot teach anything; they still count in the dev score (as the evaluator would)
TRAIN_QIDS = [q for q in queries["train"] if GOLD["train"][q]]

for split in ("train", "dev", "test"):
    Q, P = queries[split], pools[split]
    qw = np.array([len(d["text"].split()) for d in Q.values()])
    cw = np.array([len(d["text"].split()) for d in P.values()])
    line = (f"{split:5s} queries {len(Q):5d} | pool {len(P):6d} | query words median {np.median(qw):.0f} "
            f"p90 {np.percentile(qw, 90):.0f} | candidate words median {np.median(cw):.0f} "
            f"p90 {np.percentile(cw, 90):.0f}")
    if split in GOLD:
        gs = np.array([len(s) for s in GOLD[split].values()])
        line += f" | gold per query mean {gs.mean():.2f} median {np.median(gs):.0f} max {gs.max()} | >7: {(gs > 7).mean():.1%}"
    n_self = sum(q in P for q in Q)
    if n_self:
        line += f" | {n_self} queries are also in their own pool (excluded when ranking)"
    if any(d["cands"] for d in Q.values()):
        line += " | queries carry their own candidate lists (ranking restricted to them)"
    print(line)
print("pool overlap train∩dev", len(set(pools["train"]) & set(pools["dev"])),
      "| train∩test", len(set(pools["train"]) & set(pools["test"])),
      "| dev∩test", len(set(pools["dev"]) & set(pools["test"])))
print("first test query ids:", list(queries["test"])[:3])


Phase 7 files on HF: ['Phase_7_PCR/Dev/dev.json', 'Phase_7_PCR/Dev/dev_candidates.json', 'Phase_7_PCR/Test/test.json', 'Phase_7_PCR/Test/test_candidates.json', 'Phase_7_PCR/Train/train.json', 'Phase_7_PCR/Train/train_candidates.json']


Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

train.json layout: dict of 827 | first record: ['text', 'relevant_candidates']


Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

... (7 outputs left out)

test.json layout: dict of 237 | first record: ['text']


Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

test_candidates.json layout: dict of 1727 | first record: ['text']
test queries: 237 records -> 237 cases (repeated ids skipped 0)
test candidates: 1727 records -> 1727 cases (repeated ids skipped 0)
train: 5308 relevant links, 0 not in the train pool (dropped) | queries with no retrievable relevant case: 10
dev: 973 relevant links, 0 not in the dev pool (dropped) | queries with no retrievable relevant case: 0
train queries   827 | pool   4320 | query words median 4508 p90 12602 | candidate words median 3632 p90 12422 | gold per query mean 6.42 median 4 max 52 | >7: 30.0% | 827 queries are also in their own pool (excluded when ranking)
dev   queries   118 | pool   1023 | query words median 4620 p90 15333 | candidate words median 4036 p90 16599 | gold per query mean 8.25 median 5 max 92 | >7: 33.1% | 118 queries are also in their own pool (excluded when ranking)
test  queries   237 | pool   1727 | query words median 4653 p90 14065 | candidate words median 3947 p90 13491 | 237 queries ar

## Cell D — tokenisation (cached), Micro-F1@7 scorer, prediction.json writer, BM25 baseline

In [7]:
# =====================================================================================
# CELL D — [ANCILLARY] tokenisation (cached), Micro-F1@7 scorer, prediction.json writer,
#          BM25 baseline (also used as a hard-negative source in Cell E2 if you choose it)
# Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
import os, re, json, glob, time, zipfile, pickle, hashlib
import numpy as np
import scipy.sparse as sp
from collections import Counter
from tqdm.auto import tqdm

PAD_ID = SPECIAL_TOKENS.index("<pad>")   # 0
UNK_ID = SPECIAL_TOKENS.index("<unk>")   # 1
TOPK = 7                                 # the evaluator scores exactly the first 7 ids per query

# -------------------------------------------------------------------------------------
# Tokenise every query and candidate once with the c-6 BPE tokenizer -> d["ids"].
# The Python BPE is slow on long judgments, so: identical texts are encoded once, all CPU cores
# are used (fork), and the result is saved to /kaggle/working/p7_c6_tok.pkl. Attach a finished
# run's output as input next time and the cache is found and reused automatically.
# -------------------------------------------------------------------------------------
TOK_CACHE_OUT = "/kaggle/working/p7_c6_tok.pkl"

def _h(text):
    return hashlib.md5(text.encode("utf-8")).hexdigest()

def _enc(text):
    ids = tok.encode(text)
    return ids if ids else [UNK_ID]

_all_docs = [d for s in ("train", "dev", "test") for part in (queries[s], pools[s]) for d in part.values()]
_cache = {}
for _p in sorted(glob.glob("/kaggle/input/**/p7_c6_tok.pkl", recursive=True)) + [TOK_CACHE_OUT]:
    if os.path.exists(_p):
        _cache.update(pickle.load(open(_p, "rb")))
        print(f"token cache loaded from {_p}: {len(_cache)} texts")
_todo = list({_h(d["text"]): d["text"] for d in _all_docs if _h(d["text"]) not in _cache}.items())
print(f"{len(_all_docs)} documents, {len(_todo)} distinct texts still to tokenise")
if _todo:
    t0 = time.time()
    try:
        import multiprocessing as mp
        with mp.get_context("fork").Pool(os.cpu_count()) as _pool:
            _ids = list(tqdm(_pool.imap(_enc, [t for _, t in _todo], chunksize=4), total=len(_todo), desc="tokenising"))
    except Exception as e:                                       # no fork available: one process
        print("parallel tokenising failed, falling back to one process:", repr(e))
        _ids = [_enc(t) for _, t in tqdm(_todo, desc="tokenising")]
    _cache.update({h: ids for (h, _), ids in zip(_todo, _ids)})
    pickle.dump(_cache, open(TOK_CACHE_OUT, "wb"))
    print(f"tokenised in {(time.time() - t0) / 60:.1f} min, cache saved to {TOK_CACHE_OUT}")
for d in _all_docs:
    d["ids"] = _cache[_h(d["text"])]
del _cache, _todo

for s in ("train", "dev", "test"):
    for kind, part in (("queries", queries[s]), ("pool", pools[s])):
        L = np.array([len(d["ids"]) for d in part.values()])
        print(f"{s:5s} {kind:7s} c-6 tokens: median {np.median(L):.0f} p90 {np.percentile(L, 90):.0f} "
              f"max {L.max()} | windows of 512 needed (median) {np.median(np.ceil(L / 512)):.0f}")


# -------------------------------------------------------------------------------------
# Scorer: Micro-F1@7 (primary), Micro-P@7, Micro-R@7 as in the spec (gold not truncated or padded),
# plus ranking diagnostics that look past rank 7: Recall@100 and MRR.
#   pred: {qid: ranked list of candidate ids}, gold: {qid: set of ids}
# -------------------------------------------------------------------------------------
def evaluate_ranking(pred, gold, k=TOPK, show=True, name="dev"):
    tp = n_pred = n_gold = 0
    per_q, r100, rr = {}, [], []
    for q, g in gold.items():
        top = list(pred.get(q, []))[:k]
        hit = len(set(top) & g)
        tp += hit; n_pred += len(top); n_gold += len(g)
        per_q[q] = {"hits": hit, "gold": len(g), "f1": 2 * hit / (len(top) + len(g)) if (top or g) else 0.0}
        if g:
            full = list(pred.get(q, []))
            r100.append(len(set(full[:100]) & g) / len(g))
            first = next((i for i, c in enumerate(full) if c in g), None)
            rr.append(0.0 if first is None else 1.0 / (first + 1))
    P = tp / max(1, n_pred); R = tp / max(1, n_gold)
    F = 2 * P * R / (P + R) if P + R else 0.0
    res = {"micro_f1": F, "micro_p": P, "micro_r": R, "recall@100": float(np.mean(r100)) if r100 else 0.0,
           "mrr": float(np.mean(rr)) if rr else 0.0, "tp": tp, "per_query": per_q}
    if show:
        print(f"{name}: Micro-F1@{k} {F:.4f} | P@{k} {P:.4f} | R@{k} {R:.4f} | Recall@100 {res['recall@100']:.4f} "
              f"| MRR {res['mrr']:.4f} | queries {len(gold)}")
    return res


# -------------------------------------------------------------------------------------
# prediction.json writer: {query_id: [7 unique candidate ids from that query's test pool, best first]}
# for every test query, then submission.zip with prediction.json at the root.
# -------------------------------------------------------------------------------------
def allowed_pool(q, split):
    d = queries[split][q]
    ids = d["cands"] if d.get("cands") else pools[split].keys()
    return [c for c in ids if c != q and c in pools[split]]

def write_prediction(pred, out_dir="/kaggle/working", split="test"):
    os.makedirs(out_dir, exist_ok=True)
    out = {}
    for q in queries[split]:
        allowed = set(allowed_pool(q, split))
        top = [c for c in dict.fromkeys(pred.get(q, [])) if c in allowed][:TOPK]
        assert len(top) == TOPK, f"query {q}: only {len(top)} valid candidates"
        out[q] = top
    assert len(out) == len(queries[split])
    path = os.path.join(out_dir, "prediction.json")
    json.dump(out, open(path, "w"), indent=1)
    zpath = os.path.join(out_dir, "submission.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    print(f"wrote {path} ({len(out)} queries x {TOPK}) and {zpath}")
    return out


# -------------------------------------------------------------------------------------
# BM25 over lower-cased words (k1 = 1.2, b = 0.75, the textbook defaults; not tuned).
# A lexical reference point for the report, and a source of hard negatives for training.
# -------------------------------------------------------------------------------------
_TOKRE = re.compile(r"[a-z0-9]+")

class BM25:
    def __init__(self, pool_docs, k1=1.2, b=0.75):
        self.ids = list(pool_docs)
        self.vocab = {}
        rows, cols, vals, dl = [], [], [], []
        for i, c in enumerate(tqdm(self.ids, desc="BM25 index", leave=False)):
            tf = Counter(_TOKRE.findall(pool_docs[c]["text"].lower()))
            dl.append(sum(tf.values()))
            for w, n in tf.items():
                rows.append(i); cols.append(self.vocab.setdefault(w, len(self.vocab))); vals.append(n)
        tf = sp.csr_matrix((np.array(vals, np.float32), (rows, cols)), shape=(len(self.ids), len(self.vocab)))
        dl = np.array(dl, np.float32)
        df = np.bincount(tf.indices, minlength=len(self.vocab))
        idf = np.log(1 + (len(self.ids) - df + 0.5) / (df + 0.5)).astype(np.float32)
        norm = k1 * (1 - b + b * dl / dl.mean())
        tf = tf.tocoo()
        w = tf.data * (k1 + 1) / (tf.data + norm[tf.row]) * idf[tf.col]
        self.W = sp.csr_matrix((w, (tf.row, tf.col)), shape=tf.shape).T.tocsr()   # (V, Ndocs)

    def scores(self, text):
        q = Counter(w for w in _TOKRE.findall(text.lower()) if w in self.vocab)
        if not q:
            return np.zeros(len(self.ids), np.float32)
        qv = sp.csr_matrix((np.ones(len(q), np.float32), ([0] * len(q), [self.vocab[w] for w in q])),
                           shape=(1, len(self.vocab)))           # unique query terms (queries are whole judgments)
        return np.asarray((qv @ self.W).todense()).ravel()

def bm25_rank(split, top=200):
    """-> {qid: top `top` candidate ids of the split's pool by BM25}, self / non-allowed excluded."""
    bm = BM25(pools[split])
    pos = {c: i for i, c in enumerate(bm.ids)}
    out = {}
    for q, d in tqdm(queries[split].items(), desc=f"BM25 {split}"):
        s = bm.scores(d["text"])
        allowed = allowed_pool(q, split)
        idx = np.array([pos[c] for c in allowed])
        order = idx[np.argsort(-s[idx], kind="stable")[:top]]
        out[q] = [bm.ids[i] for i in order]
    return out

BM25_RANK = {"dev": bm25_rank("dev")}
print("\n-- BM25 baseline")
bm25_dev = evaluate_ranking(BM25_RANK["dev"], GOLD["dev"], name="BM25 dev")


token cache loaded from /kaggle/input/notebooks/rudraalt/phase-7-c-6-lorma/p7_c6_tok.pkl: 5819 texts
8252 documents, 0 distinct texts still to tokenise
train queries c-6 tokens: median 6053 p90 16652 max 162313 | windows of 512 needed (median) 12
train pool    c-6 tokens: median 4813 p90 16464 max 547297 | windows of 512 needed (median) 10
dev   queries c-6 tokens: median 6168 p90 20301 max 142182 | windows of 512 needed (median) 12
dev   pool    c-6 tokens: median 5332 p90 22342 max 547297 | windows of 512 needed (median) 11
test  queries c-6 tokens: median 6232 p90 18347 max 117654 | windows of 512 needed (median) 13
test  pool    c-6 tokens: median 5224 p90 18380 max 547297 | windows of 512 needed (median) 11


BM25 index:   0%|          | 0/1023 [00:00<?, ?it/s]

BM25 dev:   0%|          | 0/118 [00:00<?, ?it/s]


-- BM25 baseline
BM25 dev: Micro-F1@7 0.1012 | P@7 0.1102 | R@7 0.0935 | Recall@100 0.4073 | MRR 0.3248 | queries 118


## Cell E0 — bi-encoder retrieval plumbing (masked encoder, windows, similarity, ranking losses, GradCache, training)

In [8]:
# =====================================================================================
# CELL E0 — [ANCILLARY] shared retrieval plumbing for the c-6 Phase 7 (PCR) experiments
# Run once after Cells 0, A, B, C, D. Then run Cell E2 (LORMA).
# Needs from earlier cells: lm, cfg, device, queries, pools, GOLD, TRAIN_QIDS, PAD_ID,
#   evaluate_ranking, allowed_pool, write_prediction, bm25_rank, BM25_RANK
#
# The c-6 ENCODER is used as a bi-encoder (dual encoder with shared weights):
#   case -> token windows -> ENCODER (masked) -> one vector per window (POOL)
#        -> document = its window vectors; similarity(query, candidate) from them (SIMILARITY)
#   The decoder is not used in Phase 7.
#   Candidates of a pool are encoded once, every query is scored against all of them.
#
# Long documents (position table has 1536 rows, judgments are far longer): windows of WINDOW
#   tokens every STRIDE tokens (Phase 2/4/5/6 make_windows), DOC_REPR picks which ones stand for
#   the case ("all", "first", "last", "first_last", "spread"), up to MAX_WINDOWS.
#
# Similarity (choice in E2):
#   "cosine" / "dot" : window vectors -> one case vector (WIN_AGG "mean" or "max"), then cosine / dot
#   "maxsim"         : late interaction on L2-normalised window vectors: for every query window
#                      take its best-matching candidate window, average over query windows
#
# Training objective (choice in E2), one row per query with one sampled relevant candidate:
#   "infonce" : softmax cross-entropy over [its positive, its hard negatives, and (IN_BATCH_NEG)
#               every other candidate in the batch], logits = similarity / TAU
#   "margin"  : pairwise hinge  mean(max(0, MARGIN - s(q, pos) + s(q, neg))) over the same negatives
#   Candidates relevant to the query (other than the sampled positive) are never used as its
#   negatives (multi-relevant queries).
# Hard negatives (choice in E2): "none", "random" (from the pool), "bm25" (BM25 top HARD_TOP,
#   non-relevant), "model" (current model's top HARD_TOP, re-mined at the start of every epoch).
#
# Memory: CHUNK > 0 turns on gradient caching (GradCache): the batch is encoded in chunks without
#   gradients, the loss is taken on all embeddings at once, then each chunk is re-encoded with the
#   same dropout RNG and back-propagated. Same loss and gradients as one big batch, so in-batch
#   negatives keep working with a large BATCH_SIZE. Self-check in Cell E2 proves the gradients match.
# Nothing in this cell is a model design or a hyperparameter choice: every choice is an argument
# set in Cell E2.
# =====================================================================================
import math, time, random, os, json
import numpy as np
from collections import defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm

FEAT_DIM = cfg.d_model          # 560 for c-6
MAX_POS = cfg.max_seq_len       # 1536: longest encoder window
GRAD_CHECKPOINT = False         # set in Cell E2
_AUTOCAST = torch.cuda.is_available()


# -------------------------------------------------------------------------------------
# Encoder with a key-padding mask (same maths as Phase 1 EncoderSelfAttention / EncoderBlock;
# Phase 1 has no padding mask, needed here to batch windows of different lengths)
# -------------------------------------------------------------------------------------
def _heads(t, B, T, H, D):
    return t.view(B, T, H, D).transpose(1, 2)

def _scale(attn):
    return getattr(attn, "attention_scale", None) or getattr(attn, "scale", None) or 1.0 / math.sqrt(attn.d_head)

def enc_self_attn(attn, x, key_mask):
    B, T, C = x.shape
    H, D = attn.n_heads, attn.d_head
    q, k, v = [_heads(t, B, T, H, D) for t in attn.qkv_proj(x).chunk(3, dim=-1)]
    m = None if key_mask is None else key_mask[:, None, None, :]
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=m, scale=_scale(attn),
                                         dropout_p=attn.attn_dropout.p if attn.training else 0.0)
    return attn.resid_dropout(attn.out_proj(out.transpose(1, 2).contiguous().view(B, T, C)))

def _enc_block(blk, x, key_mask):
    x = x + enc_self_attn(blk.self_attn, blk.ln1(x), key_mask)
    return x + blk.ffn(blk.ln2(x))

def encoder_forward(backbone, idx, key_mask=None):
    """idx (N, T) padded with PAD_ID, key_mask (N, T) bool (True = real token) -> (N, T, 560)."""
    c = backbone.core
    assert idx.size(1) <= MAX_POS, f"encoder input of {idx.size(1)} tokens > {MAX_POS}"
    x = c.tok_emb(idx) + c.get_positional_embeddings(idx.size(1), idx.device)
    ck = GRAD_CHECKPOINT and torch.is_grad_enabled() and backbone.training
    for blk in c.encoder_layers:
        x = checkpoint(_enc_block, blk, x, key_mask, use_reentrant=False) if ck else _enc_block(blk, x, key_mask)
    return c.encoder_ln(x)


# -------------------------------------------------------------------------------------
# Which encoder linears can be wrapped, by short name
# -------------------------------------------------------------------------------------
_WHERE = {"enc_qkv": ("self_attn", "qkv_proj"), "enc_out": ("self_attn", "out_proj"),
          "enc_ffn_in": ("ffn", "net", "0"), "enc_ffn_out": ("ffn", "net", "2")}

def all_target_names(backbone=None):
    return tuple(_WHERE)

def model_linears(backbone, which):
    """Yield (parent, attr, module) for each chosen linear in every encoder block."""
    for w in which:
        assert w in _WHERE, f"unknown target {w}; choose from {tuple(_WHERE)}"
        path = _WHERE[w]
        for blk in backbone.core.encoder_layers:
            parent = blk
            for p in path[:-1]:
                parent = getattr(parent, p)
            yield parent, path[-1], getattr(parent, path[-1])

def wrap_linears(backbone, which, make_wrapper):
    todo = list(model_linears(backbone, which))
    for parent, attr, lin in todo:
        assert isinstance(lin, nn.Linear), f"{attr} is already wrapped (re-run Cell A and E0 to start fresh)"
        setattr(parent, attr, make_wrapper(lin))
    return len(todo)

def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / total {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot


# -------------------------------------------------------------------------------------
# Windows (Phase 2/4/5/6 make_windows + select_windows, unchanged)
# -------------------------------------------------------------------------------------
DOC_REPRS = ("all", "first", "last", "first_last", "spread")
POOLS = ("mean", "max", "first")
WIN_AGGS = ("mean", "max")
SIMILARITIES = ("cosine", "dot", "maxsim")
LOSSES = ("infonce", "margin")
HARD_SOURCES = ("none", "random", "bm25", "model")
POS_SAMPLINGS = ("one_per_query", "all_pairs")

def make_windows(n, window, stride):
    if n <= window:
        return [(0, n)]
    starts = list(range(0, n - window, stride)) + [n - window]
    return [(s, s + window) for s in starts]

def select_windows(wins, doc_repr, max_windows):
    n = len(wins)
    if doc_repr == "all" or n <= max_windows:
        return wins
    if doc_repr == "first":
        return wins[:max_windows]
    if doc_repr == "last":
        return wins[-max_windows:]
    if doc_repr == "first_last":
        h = (max_windows + 1) // 2
        return wins[:h] + wins[n - (max_windows - h):] if max_windows > h else wins[:h]
    if doc_repr == "spread":
        idx = sorted({round(i * (n - 1) / max(1, max_windows - 1)) for i in range(max_windows)})
        return [wins[i] for i in idx]
    raise ValueError(doc_repr)

def build_pieces(docs, window, stride, doc_repr, max_windows, desc="windows"):
    """{case_id: d} -> {case_id: [token lists, one per selected window]}."""
    assert window <= MAX_POS and stride <= window and doc_repr in DOC_REPRS
    assert doc_repr == "all" or max_windows, "set MAX_WINDOWS for this DOC_REPR"
    out = {k: [d["ids"][s:e] for s, e in select_windows(make_windows(len(d["ids"]), window, stride),
                                                           doc_repr, max_windows)]
           for k, d in tqdm(docs.items(), desc=desc, leave=False)}
    nw = np.array([len(v) for v in out.values()])
    seen = sum(sum(map(len, v)) for v in out.values()) / max(1, sum(len(d["ids"]) for d in docs.values()))
    print(f"{desc}: {len(out)} cases | windows per case mean {nw.mean():.2f} max {nw.max()} | tokens seen {seen:.1%}")
    return out


# -------------------------------------------------------------------------------------
# Bi-encoder: list of cases (each a list of windows) -> window vectors E (N, W, D) + mask M (N, W)
# -------------------------------------------------------------------------------------
def _pad(rows):
    T = max(len(r) for r in rows)
    idx = torch.tensor([r + [PAD_ID] * (T - len(r)) for r in rows], dtype=torch.long, device=device)
    L = torch.tensor([len(r) for r in rows], device=device)
    mask = torch.arange(T, device=device)[None, :] < L[:, None]
    return idx, (None if bool(mask.all()) else mask), mask

class PCREncoder(nn.Module):
    def __init__(self, backbone, pool):
        super().__init__()
        assert pool in POOLS
        self.backbone, self.pool = backbone, pool

    def forward(self, cases):
        flat = [w for c in cases for w in c]
        idx, key_mask, tmask = _pad(flat)
        h = encoder_forward(self.backbone, idx, key_mask)            # (Nw, T, D)
        m = tmask.unsqueeze(-1).to(h.dtype)
        if self.pool == "mean":
            v = (h * m).sum(1) / m.sum(1).clamp_min(1)
        elif self.pool == "max":
            v = h.masked_fill(~tmask.unsqueeze(-1), torch.finfo(h.dtype).min).max(1).values
        else:                                                        # "first": first token of each window
            v = h[:, 0]
        W = max(len(c) for c in cases)
        E = v.new_zeros(len(cases), W, v.size(-1))
        M = torch.zeros(len(cases), W, dtype=torch.bool, device=v.device)
        i = 0
        for r, c in enumerate(cases):
            E[r, :len(c)] = v[i:i + len(c)]; M[r, :len(c)] = True
            i += len(c)
        return E, M


def similarity(qE, qM, cE, cM, sim, win_agg):
    """(Nq, Wq, D), (Nq, Wq), (Nc, Wc, D), (Nc, Wc) -> scores (Nq, Nc), fp32."""
    qE, cE = qE.float(), cE.float()
    if sim == "maxsim":
        qn, cn = F.normalize(qE, dim=-1), F.normalize(cE, dim=-1)
        s = torch.einsum("qid,cjd->qcij", qn, cn)                    # window-to-window cosine
        s = s.masked_fill(~cM[None, :, None, :], -1e4).max(-1).values  # best candidate window
        qm = qM[:, None, :].float()
        return (s * qm).sum(-1) / qm.sum(-1).clamp_min(1)            # mean over query windows
    def agg(E, M):
        if win_agg == "mean":
            m = M.unsqueeze(-1).float()
            return (E * m).sum(1) / m.sum(1).clamp_min(1)
        return E.masked_fill(~M.unsqueeze(-1), -1e4).max(1).values
    qv, cv = agg(qE, qM), agg(cE, cM)
    if sim == "cosine":
        qv, cv = F.normalize(qv, dim=-1), F.normalize(cv, dim=-1)
    return qv @ cv.T


# -------------------------------------------------------------------------------------
# Encoding a whole pool, ranking, prediction
# -------------------------------------------------------------------------------------
@torch.no_grad()
def encode_cases(enc, pieces, ids, batch_windows, desc="encode"):
    """-> E (N, Wmax, D) fp16, M (N, Wmax) for `ids` in order. Batches by window count (less padding)."""
    enc.eval()
    order = sorted(range(len(ids)), key=lambda i: sum(map(len, pieces[ids[i]])))
    W = max(len(pieces[k]) for k in ids)
    E = torch.zeros(len(ids), W, FEAT_DIM, dtype=torch.float16, device=device)
    M = torch.zeros(len(ids), W, dtype=torch.bool, device=device)
    i, pbar = 0, tqdm(total=len(ids), desc=desc, leave=False)
    while i < len(order):
        j, nw = i, 0
        while j < len(order) and (j == i or nw + len(pieces[ids[order[j]]]) <= batch_windows):
            nw += len(pieces[ids[order[j]]]); j += 1
        sel = order[i:j]
        with torch.autocast("cuda", dtype=torch.float16, enabled=_AUTOCAST):
            e, m = enc([pieces[ids[t]] for t in sel])
        E[sel, :e.size(1)] = e.half(); M[sel, :m.size(1)] = m
        pbar.update(j - i); i = j
    pbar.close()
    return E, M

@torch.no_grad()
def rank_split(enc, split, q_pieces, c_pieces, sim, win_agg, batch_windows, top=200, q_ids=None, return_scores=False):
    """-> {qid: top `top` allowed candidate ids, best first} (and their scores if return_scores)."""
    cids = list(pools[split])
    pos = {c: i for i, c in enumerate(cids)}
    q_ids = list(queries[split]) if q_ids is None else q_ids
    cE, cM = encode_cases(enc, c_pieces, cids, batch_windows, desc=f"encode {split} pool")
    qE, qM = encode_cases(enc, q_pieces, q_ids, batch_windows, desc=f"encode {split} queries")
    out, sc = {}, {}
    qb = 8 if sim == "maxsim" else 256
    cb = 2048 if sim == "maxsim" else len(cids)
    for s in tqdm(range(0, len(q_ids), qb), desc=f"rank {split}", leave=False):
        S = torch.cat([similarity(qE[s:s + qb], qM[s:s + qb], cE[t:t + cb], cM[t:t + cb], sim, win_agg)
                       for t in range(0, len(cids), cb)], dim=1)
        for r, q in enumerate(q_ids[s:s + qb]):
            row = S[r]
            d = queries[split][q]
            if d.get("cands") or q in pos:                           # restrict to allowed candidates
                keep = torch.zeros_like(row, dtype=torch.bool)
                keep[[pos[c] for c in allowed_pool(q, split)]] = True
                row = row.masked_fill(~keep, -float("inf"))
            v, i = torch.topk(row, min(top, row.numel()))
            ok = torch.isfinite(v)
            out[q] = [cids[j] for j in i[ok].tolist()]
            sc[q] = v[ok].tolist()
    del cE, cM, qE, qM
    return (out, sc) if return_scores else out


# -------------------------------------------------------------------------------------
# Training batches
# -------------------------------------------------------------------------------------
def make_epoch_rows(pos_sampling, gold):
    """-> list of (qid, positive cid) for one epoch, shuffled."""
    rows = []
    for q in TRAIN_QIDS:
        g = sorted(gold[q])
        rows += [(q, random.choice(g))] if pos_sampling == "one_per_query" else [(q, c) for c in g]
    random.shuffle(rows)
    return rows

def pick_negatives(q, n_hard, source, hard_rank, hard_top, gold):
    if not n_hard or source == "none":
        return []
    if source == "random":
        pool_ids = TRAIN_POOL_IDS
        out = set()
        while len(out) < n_hard:
            c = random.choice(pool_ids)
            if c != q and c not in gold[q]:
                out.add(c)
        return list(out)
    cands = [c for c in hard_rank[q][:hard_top] if c not in gold[q] and c != q]
    return random.sample(cands, min(n_hard, len(cands)))

def batch_mask(rows, cand_ids, own_negs, gold, in_batch, dev):
    """rows[i] = (qid, pos); cand_ids = unique candidate ids in the batch (columns);
    own_negs[i] = set of column indices that are query i's own hard negatives.
    -> allowed (B, C) bool (positive + usable negatives), target (B,) column of each positive."""
    col = {c: j for j, c in enumerate(cand_ids)}
    B, C = len(rows), len(cand_ids)
    target = torch.tensor([col[p] for _, p in rows], device=dev)
    allowed = torch.zeros(B, C, dtype=torch.bool, device=dev)
    for i, (q, p) in enumerate(rows):
        if in_batch:
            allowed[i] = True
        else:
            allowed[i, list(own_negs[i])] = True
        for c in gold[q]:                                            # other relevant cases are not negatives
            if c in col and c != p:
                allowed[i, col[c]] = False
        if q in col:
            allowed[i, col[q]] = False
        allowed[i, col[p]] = True
    return allowed, target

def score_loss(S, allowed, target, loss_type, tau, margin):
    """Ranking loss on a (B, C) similarity matrix."""
    if loss_type == "infonce":
        return F.cross_entropy((S / tau).masked_fill(~allowed, -1e4), target)
    B = S.size(0)
    s_pos = S.gather(1, target[:, None])
    neg = allowed.clone(); neg[torch.arange(B), target] = False
    h = F.relu(margin - s_pos + S) * neg
    return (h.sum(1) / neg.sum(1).clamp_min(1)).mean()

def batch_loss(qE, qM, cE, cM, rows, cand_ids, own_negs, gold, sim, win_agg, loss_type, tau, margin, in_batch):
    S = similarity(qE, qM, cE, cM, sim, win_agg)                     # (B, C)
    allowed, target = batch_mask(rows, cand_ids, own_negs, gold, in_batch, S.device)
    return score_loss(S, allowed, target, loss_type, tau, margin)


def _rng_state():
    return (torch.get_rng_state(), torch.cuda.get_rng_state() if torch.cuda.is_available() else None)

def _set_rng(st):
    torch.set_rng_state(st[0])
    if st[1] is not None:
        torch.cuda.set_rng_state(st[1])

def step_backward(enc, q_cases, c_cases, loss_fn, chunk, scaler):
    """Forward + backward for one batch. chunk = cases per forward (None/0 = all at once).
    Returns the loss value. With chunk, GradCache: identical loss/gradients, much less memory."""
    cases = q_cases + c_cases
    nq = len(q_cases)
    ac = dict(device_type="cuda", dtype=torch.float16, enabled=_AUTOCAST)
    if not chunk or chunk >= len(cases):
        with torch.autocast(**ac):
            E, M = enc(cases)
        loss = loss_fn(E[:nq], M[:nq], E[nq:], M[nq:])
        scaler.scale(loss).backward()
        return loss.item()
    parts = [cases[i:i + chunk] for i in range(0, len(cases), chunk)]
    states, Es, Ms = [], [], []
    with torch.no_grad():
        for p in parts:
            states.append(_rng_state())
            with torch.autocast(**ac):
                e, m = enc(p)
            Es.append(e.float()); Ms.append(m)
    W = max(e.size(1) for e in Es)
    E = torch.cat([F.pad(e, (0, 0, 0, W - e.size(1))) for e in Es]).requires_grad_(True)
    M = torch.cat([F.pad(m, (0, W - m.size(1))) for m in Ms])
    loss = loss_fn(E[:nq], M[:nq], E[nq:], M[nq:])
    loss.backward()
    G = E.grad
    after = _rng_state()
    i = 0
    for p, st in zip(parts, states):
        _set_rng(st)
        with torch.autocast(**ac):
            e, _ = enc(p)
        g = G[i:i + len(p), :e.size(1)]
        scaler.scale((e.float() * g).sum()).backward()
        i += len(p)
    _set_rng(after)
    return loss.item()


# -------------------------------------------------------------------------------------
# Training loop: AdamW on trainable params (fp16 autocast + GradScaler), best epoch by dev Micro-F1@7.
# Saves best_trainable.pt and last_checkpoint.pt (resume: trainable, best, opt, scaler, sched, epoch, log).
# -------------------------------------------------------------------------------------
def trainable_state(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

def train_model(enc, pieces, lr, epochs, batch_size, weight_decay, sim, win_agg, loss_type, tau, margin,
                in_batch, hard_source, n_hard, hard_top, pos_sampling, grad_clip=None, lr_lambda=None,
                chunk=None, batch_windows=64, save_path="best_trainable.pt", last_path=None,
                resume_from=None, max_epoch_minutes=None):
    """pieces = {"train_q", "train_c", "dev_q", "dev_c"}: case id -> windows."""
    params = [p for p in enc.parameters() if p.requires_grad]
    assert params, "no trainable parameters"
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=_AUTOCAST)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda) if lr_lambda else None
    best, log, start = -1.0, [], 0
    if resume_from:
        ck = torch.load(resume_from, map_location="cpu", weights_only=False)
        enc.load_state_dict(ck["trainable"], strict=False)
        opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        if sched and ck.get("sched"):
            sched.load_state_dict(ck["sched"])
        best, log, start = ck["best"], ck["log"], ck["epoch"]
        torch.save(ck["best_trainable"], save_path)
        print(f"resumed after epoch {start} (best dev Micro-F1@7 {best:.4f})")
        del ck
    gold = GOLD["train"]
    hard_rank = None
    if hard_source == "bm25" and n_hard:
        if "train" not in BM25_RANK:
            BM25_RANK["train"] = bm25_rank("train", top=max(200, hard_top))
        hard_rank = BM25_RANK["train"]
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    for ep in range(start, epochs):
        if hard_source == "model" and n_hard:
            hard_rank = rank_split(enc, "train", pieces["train_q"], pieces["train_c"], sim, win_agg,
                                   batch_windows, top=hard_top + 50, q_ids=TRAIN_QIDS)
        enc.train()
        rows = make_epoch_rows(pos_sampling, gold)
        batches = [rows[i:i + batch_size] for i in range(0, len(rows), batch_size)]
        t0, tot, seen = time.time(), 0.0, 0
        pbar = tqdm(batches, desc=f"epoch {ep + 1}/{epochs}")
        for batch in pbar:
            negs = [pick_negatives(q, n_hard, hard_source, hard_rank, hard_top, gold) for q, _ in batch]
            cand_ids = list(dict.fromkeys([p for _, p in batch] + [c for n in negs for c in n]))
            col = {c: j for j, c in enumerate(cand_ids)}
            own = [{col[c] for c in n} for n in negs]
            q_cases = [pieces["train_q"][q] for q, _ in batch]
            c_cases = [pieces["train_c"][c] for c in cand_ids]
            loss_fn = lambda qE, qM, cE, cM: batch_loss(qE, qM, cE, cM, batch, cand_ids, own, gold, sim, win_agg,
                                                        loss_type, tau, margin, in_batch)
            opt.zero_grad(set_to_none=True)
            l = step_backward(enc, q_cases, c_cases, loss_fn, chunk, scaler)
            if grad_clip:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, grad_clip)
            scaler.step(opt); scaler.update()
            if sched:
                sched.step()
            tot += l; seen += 1
            pbar.set_postfix(loss=f"{tot / seen:.4f}", lr=f"{opt.param_groups[0]['lr']:.2e}")
            if max_epoch_minutes and time.time() - t0 > 60 * max_epoch_minutes:
                print(f"  time cap: stopped epoch after {seen}/{len(batches)} batches")
                break
        dev_rank = rank_split(enc, "dev", pieces["dev_q"], pieces["dev_c"], sim, win_agg, batch_windows)
        r = evaluate_ranking(dev_rank, GOLD["dev"], show=False)
        row = {"epoch": ep + 1, "train_loss": tot / max(1, seen), "batches": seen,
               "minutes": (time.time() - t0) / 60, **{k: r[k] for k in ("micro_f1", "micro_p", "micro_r", "recall@100", "mrr")}}
        log.append(row)
        improved = r["micro_f1"] > best
        if improved:
            best = r["micro_f1"]
            torch.save(trainable_state(enc), save_path)
        print(f"epoch {ep + 1}: train loss {row['train_loss']:.4f} | dev Micro-F1@7 {r['micro_f1']:.4f} "
              f"(P {r['micro_p']:.4f} R {r['micro_r']:.4f}) | R@100 {r['recall@100']:.4f} | MRR {r['mrr']:.4f}"
              f"{' *best*' if improved else ''} | {row['minutes']:.1f} min")
        if last_path:
            torch.save({"trainable": trainable_state(enc), "best_trainable": torch.load(save_path),
                        "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                        "sched": sched.state_dict() if sched else None, "epoch": ep + 1, "best": best,
                        "log": log}, last_path)
    return log, best


# -------------------------------------------------------------------------------------
# Error breakdown helpers (numbers for the report; the analysis itself is yours)
# -------------------------------------------------------------------------------------
def breakdown(res, split="dev"):
    """Micro-F1@7 by gold-set size and by query length (c-6 tokens)."""
    pq = res["per_query"]
    def show(title, key, edges):
        print(title)
        for lo, hi in zip(edges[:-1], edges[1:]):
            qs = [q for q in pq if lo <= key(q) < hi]
            if qs:
                tp = sum(pq[q]["hits"] for q in qs); ng = sum(pq[q]["gold"] for q in qs)
                P = tp / (TOPK * len(qs)); R = tp / max(1, ng)
                print(f"  [{lo}, {hi}): {len(qs):4d} queries | Micro-F1@7 {2*P*R/(P+R) if P+R else 0:.4f}")
    show("by gold-set size:", lambda q: pq[q]["gold"], [0, 1, 3, 6, 8, 12, 10**9])
    L = {q: len(queries[split][q]["ids"]) for q in pq}
    e = [0] + [int(np.percentile(list(L.values()), p)) for p in (25, 50, 75)] + [10**9]
    show("by query length (quartiles, c-6 tokens):", lambda q: L[q], sorted(set(e)))

def _self_checks():
    lm.eval()
    q = next(iter(queries["dev"].values()))
    src = torch.tensor([q["ids"][:64]], device=device)
    with torch.no_grad():
        e1 = (encoder_forward(lm, src) - lm.encode(src)).abs().max().item()
        short = src[:, :40]
        padded = torch.cat([short, torch.full((1, 24), PAD_ID, device=device)], dim=1)
        mask = torch.arange(64, device=device)[None, :] < 40
        e2 = (encoder_forward(lm, padded, mask)[:, :40] - lm.encode(short)).abs().max().item()
    print(f"self-checks max |diff|: encoder {e1:.1e} | padded encoder {e2:.1e} (both should be ~0)")
    assert max(e1, e2) < 1e-3, "masked encoder differs from the Phase 1 model"

_self_checks()
TRAIN_POOL_IDS = list(pools["train"])


self-checks max |diff|: encoder 9.5e-07 | padded encoder 9.5e-07 (both should be ~0)


## Cell K1 — rebuild the LORMA teacher, cache its vectors

In [9]:
# =====================================================================================
# CELL K1 — [ANCILLARY] rebuild the TEACHER: your Phase 7 c-6 LORMA retriever
#
# Run after Cells 0, A, B, C, D and E0 of THIS notebook (the same cells as the LORMA notebook).
# Attach two inputs (Add Input):
#   1. your Phase 1 c-6 notebook output (ROOT in Cell A) -> Phase 1 weights, used for the teacher
#      backbone AND for the student
#   2. your Phase 7 c-6 LORMA notebook output holding c6_lorma_pcr_<...>/teacher_config.json +
#      best_trainable.pt (+ teacher_rankings.json). Optionally also p7_c6_tok.pkl (skips tokenising).
# TEACHER_DIR is found automatically if left None.
#
# Teacher = Phase 1 c-6 encoder (8 blocks) + your LORMA (I + BA)W on the targets saved in
# teacher_config.json, loaded with the trained A/B matrices, same windows / pooling / similarity
# as in the LORMA run. It stays frozen. Its window vectors for the train queries and the train
# pool are computed ONCE here and cached, so teacher scores cost nothing during KD training.
# Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
import glob, zipfile

TEACHER_DIR = None   # e.g. "/kaggle/input/<...>/c6_lorma_pcr_cosine_infonce_bm25" (None = search /kaggle/input)
ENC_BATCH_WINDOWS = 64   # windows per forward when encoding (lower it on OutOfMemoryError)

if TEACHER_DIR is None:
    _found = [p for p in sorted(glob.glob("/kaggle/input/**/teacher_config.json", recursive=True))
              if "pcr" in json.load(open(p)).get("run", "")]
    print("Phase 7 teacher_config.json found at:", _found)
    assert _found, "no Phase 7 teacher_config.json under /kaggle/input: attach your Phase 7 c-6 LORMA notebook output"
    assert len(_found) == 1, "more than one teacher_config.json found: set TEACHER_DIR to the folder you want"
    TEACHER_DIR = os.path.dirname(_found[0])
print("TEACHER_DIR =", TEACHER_DIR, "|", sorted(os.listdir(TEACHER_DIR)))

# A .pt file is a zip archive inside, and Kaggle datasets unpack zips on upload: best_trainable.pt
# may arrive as a FOLDER (best_trainable/.../data.pkl + data/ + version). If so, pack it back.
def find_weights(folder, stem="best_trainable"):
    for cand in (f"{folder}/{stem}.pt", f"{folder}/{stem}.pt/{stem}.pt"):
        if os.path.isfile(cand):
            return cand
    pk = [q for q in glob.glob(f"{folder}/**/data.pkl", recursive=True) if stem in q]
    assert pk, f"no {stem}.pt (file or unpacked folder) under {folder}"
    inner = os.path.dirname(pk[0])
    root = os.path.dirname(inner)
    out = f"/kaggle/working/{stem}_repacked.pt"
    with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_STORED) as z:
        for dp, _, fs in os.walk(inner):
            for fn in sorted(fs):
                full = os.path.join(dp, fn)
                z.write(full, arcname=os.path.relpath(full, root))
    print(f"{stem}.pt was unpacked by Kaggle ({inner}); repacked -> {out}")
    return out

T_WEIGHTS = find_weights(TEACHER_DIR)
T_CFG = json.load(open(f"{TEACHER_DIR}/teacher_config.json"))
assert "LORMA" in T_CFG["method"], f"teacher run is {T_CFG['method']}, not LORMA"
print("teacher run:", T_CFG["run"], "| method", T_CFG["method"], "| rank", T_CFG["lorma_rank"],
      "\n  targets", T_CFG["lorma_targets"], "\n  build", T_CFG["build"], "| pool", T_CFG["pool"],
      "| ranking", T_CFG["ranking"])

BUILD = dict(T_CFG["build"])          # student sees the SAME windows as the teacher
RANKING = dict(T_CFG["ranking"])      # and scores with the same similarity, so scores are comparable


# -------------------------------------------------------------------------------------
# Your LORMA layer, copied unchanged from the LORMA notebook's Cell E2: W' = (I + BA) W
# (multiplicative LORMA, not LoRA; the teacher's trained A/B only load into exactly this class)
# -------------------------------------------------------------------------------------
class LORMALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int):
        super().__init__()
        self.base = base
        self.in_features = base.in_features
        self.out_features = base.out_features
        self.rank = rank
        # Freeze pretrained weights
        for p in self.base.parameters():
            p.requires_grad = False

        # A: (rank, out_features), since A operates on Wx
        # B: (out_features, rank)
        self.lorma_A = nn.Parameter(
            torch.empty(self.rank, self.out_features)
        )
        self.lorma_B = nn.Parameter(
            torch.empty(self.out_features, self.rank)
        )

        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_B)

    def forward(self, x):
        # Compute pretrained linear output without bias: Wx
        wx = F.linear(x, self.base.weight, bias=None)

        # Low-rank transformation of Wx: BA(Wx)
        lorma_out = F.linear(
            F.linear(wx, self.lorma_A),
            self.lorma_B
        )

        # Return Wx + BA(Wx) + b   ==  (I + BA) W x + b
        if self.base.bias is not None:
            return wx + lorma_out + self.base.bias

        return wx + lorma_out

# check that it really is multiplicative LORMA: (I + BA) W, not W + BA
_t = LORMALinear(nn.Linear(560, 1680, bias=False), rank=4)
_x = torch.randn(2, 7, 560)
with torch.no_grad():
    nn.init.normal_(_t.lorma_B, std=0.02)
    assert torch.allclose(_t(_x), _x @ ((torch.eye(1680) + _t.lorma_B @ _t.lorma_A) @ _t.base.weight).T, atol=1e-4)
del _t, _x


# -------------------------------------------------------------------------------------
# Rebuild: lm (Cell A) + LORMA on the teacher's targets, load the trained A/B, freeze everything
# -------------------------------------------------------------------------------------
for p in lm.parameters():
    p.requires_grad = False
_n = wrap_linears(lm, T_CFG["lorma_targets"], lambda lin: LORMALinear(lin, T_CFG["lorma_rank"]).to(device))
teacher = PCREncoder(lm, T_CFG["pool"]).to(device)
_sd = torch.load(T_WEIGHTS, map_location="cpu")
_r = teacher.load_state_dict(_sd, strict=False)
_need = {n for n, p in teacher.named_parameters() if p.requires_grad}
assert not _r.unexpected_keys and _need <= set(_sd), \
    f"teacher weights don't fit: unexpected {_r.unexpected_keys[:5]} missing {sorted(_need - set(_sd))[:5]}"
assert all("lorma_" in k for k in _sd), "best_trainable.pt holds non-LORMA tensors: is this the LORMA run?"
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False
print(f"teacher: wrapped {_n} linears, loaded {len(_sd)} LORMA tensors "
      f"({sum(v.numel() for v in _sd.values()) / 1e6:.3f}M)")
del _sd

def retrieval_params(backbone):
    """Parameters used for retrieval: embeddings + encoder blocks + encoder LayerNorm (decoder unused)."""
    c = backbone.core
    mods = [c.tok_emb, c.pos_emb, c.encoder_layers, c.encoder_ln]
    return sum(p.numel() for m in mods for p in m.parameters())

T_PARAMS = retrieval_params(lm)
print(f"teacher retrieval parameters {T_PARAMS / 1e6:.2f}M ({len(lm.core.encoder_layers)} encoder blocks)")

pieces = {"train_q": build_pieces({q: queries["train"][q] for q in TRAIN_QIDS}, **BUILD, desc="train queries"),
          "train_c": build_pieces(pools["train"], **BUILD, desc="train pool"),
          "dev_q": build_pieces(queries["dev"], **BUILD, desc="dev queries"),
          "dev_c": build_pieces(pools["dev"], **BUILD, desc="dev pool"),
          "test_q": build_pieces(queries["test"], **BUILD, desc="test queries"),
          "test_c": build_pieces(pools["test"], **BUILD, desc="test pool")}

# Teacher on dev (+ timing, for the size / speed comparison)
if torch.cuda.is_available():
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
_t0 = time.time()
T_dev_rank = rank_split(teacher, "dev", pieces["dev_q"], pieces["dev_c"], **RANKING,
                        batch_windows=ENC_BATCH_WINDOWS, top=100)
T_dev_s = time.time() - _t0
T_dev_mem = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
print("\n-- TEACHER on dev")
T_res = evaluate_ranking(T_dev_rank, GOLD["dev"], name="teacher dev")
print(f"teacher dev ranking {T_dev_s:.0f} s | peak memory {T_dev_mem:.2f} GB")

# Cache the teacher's window vectors for the train queries and the train pool (fp16)
T_Q_IDS, T_C_IDS = list(pieces["train_q"]), list(pieces["train_c"])
T_QE, T_QM = encode_cases(teacher, pieces["train_q"], T_Q_IDS, ENC_BATCH_WINDOWS, desc="teacher train queries")
T_CE, T_CM = encode_cases(teacher, pieces["train_c"], T_C_IDS, ENC_BATCH_WINDOWS, desc="teacher train pool")
T_QI = {q: i for i, q in enumerate(T_Q_IDS)}
T_CI = {c: i for i, c in enumerate(T_C_IDS)}
print(f"teacher vectors cached: queries {tuple(T_QE.shape)}, pool {tuple(T_CE.shape)} "
      f"({(T_QE.numel() + T_CE.numel()) * 2 / 2**20:.0f} MB)")

def teacher_scores(rows, cand_ids):
    """(B, C) teacher similarity for the batch queries x batch candidates (no encoder pass)."""
    qi = [T_QI[q] for q, _ in rows]; ci = [T_CI[c] for c in cand_ids]
    return similarity(T_QE[qi], T_QM[qi], T_CE[ci], T_CM[ci], **RANKING)

# Teacher's train rankings (hard negatives for the student if HARD_NEG == "teacher")
_tr = f"{TEACHER_DIR}/teacher_rankings.json"
if os.path.isfile(_tr) and set(TRAIN_QIDS) <= set(json.load(open(_tr)).get("train", {})):
    T_TRAIN_RANK = {q: v["ids"] for q, v in json.load(open(_tr))["train"].items()}
    print("teacher train rankings: taken from teacher_rankings.json")
else:
    T_TRAIN_RANK = rank_split(teacher, "train", pieces["train_q"], pieces["train_c"], **RANKING,
                              batch_windows=ENC_BATCH_WINDOWS, top=200, q_ids=TRAIN_QIDS)


Phase 7 teacher_config.json found at: ['/kaggle/input/notebooks/rudraalt/phase-7-c-6-lorma/c6_lorma_pcr_cosine_infonce_bm25/teacher_config.json']
TEACHER_DIR = /kaggle/input/notebooks/rudraalt/phase-7-c-6-lorma/c6_lorma_pcr_cosine_infonce_bm25 | ['best_trainable.pt', 'last_checkpoint.pt', 'prediction.json', 'submission.zip', 'teacher_config.json', 'teacher_rankings.json', 'val_log.json']
teacher run: c6_lorma_pcr_cosine_infonce_bm25 | method LORMA (I+BA)W | rank 80 
  targets ['enc_qkv', 'enc_out', 'enc_ffn_in', 'enc_ffn_out'] 
  build {'window': 512, 'stride': 256, 'doc_repr': 'spread', 'max_windows': 8} | pool mean | ranking {'sim': 'cosine', 'win_agg': 'mean'}
teacher: wrapped 32 linears, loaded 64 LORMA tensors (4.864M)
teacher retrieval parameters 30.92M (8 encoder blocks)


train queries:   0%|          | 0/817 [00:00<?, ?it/s]

train queries: 817 cases | windows per case mean 7.68 max 8 | tokens seen 42.6%


train pool:   0%|          | 0/4320 [00:00<?, ?it/s]

train pool: 4320 cases | windows per case mean 7.27 max 8 | tokens seen 44.0%


dev queries:   0%|          | 0/118 [00:00<?, ?it/s]

... (10 outputs left out)


-- TEACHER on dev
teacher dev: Micro-F1@7 0.2379 | P@7 0.2591 | R@7 0.2199 | Recall@100 0.7616 | MRR 0.5213 | queries 118
teacher dev ranking 62 s | peak memory 1.90 GB


teacher train queries:   0%|          | 0/817 [00:00<?, ?it/s]

teacher train pool:   0%|          | 0/4320 [00:00<?, ?it/s]

teacher vectors cached: queries (817, 8, 560), pool (4320, 8, 560) (44 MB)
teacher train rankings: taken from teacher_rankings.json


## Cell K2 — student + KD loss (fill section [H])

In [10]:
# =====================================================================================
# CELL K2 — KNOWLEDGE DISTILLATION: the student retriever and the KD loss
#   TEACHER = your Phase 7 c-6 LORMA retriever (Cell K1), frozen.
#   STUDENT = Phase 1 c-6 keeping KEEP_ENC_LAYERS encoder blocks, no decoder.
#   loss = (1 - KD_WEIGHT) * gold ranking loss + KD_WEIGHT * KD loss vs teacher scores
# =====================================================================================

# -------------------------------------------------------------------------------------
# [H] Hyperparameters and design choices — YOURS
# -------------------------------------------------------------------------------------
# student size
KEEP_ENC_LAYERS = [0, 2, 4, 6]
# what trains in the student
S_TRAIN         = 'full'
S_LORMA_RANK    = None   # unused with S_TRAIN = "full"
S_LORMA_TARGETS = None   # unused with S_TRAIN = "full"

# distillation
KD_TYPE         = 'listwise'
KD_WEIGHT       = 0.5
KD_TEMPERATURE  = 2.0

# gold-label ranking loss and negatives (as in the LORMA notebook)
S_LOSS          = 'infonce'
S_TAU           = 0.05
S_MARGIN        = 0.2
IN_BATCH_NEG    = True
HARD_NEG        = 'bm25'
N_HARD          = 4
HARD_TOP        = 50
POS_SAMPLING    = 'one_per_query'

# training
S_LR            = 1e-05
S_EPOCHS        = 10
S_BATCH_SIZE    = 4
S_WEIGHT_DECAY  = 0.0
S_GRAD_CLIP     = 1.0
S_LR_LAMBDA     = None
S_MAX_EPOCH_MINUTES = None
SEED            = 42

# [ANCILLARY] GPU memory: same maths and same updates, only slower.
GRAD_CHECKPOINT = True
CHUNK           = 4
# [ANCILLARY] continue an earlier student run: path to its last_checkpoint.pt. None = fresh start.
S_RESUME_FROM   = None

_required = ["KEEP_ENC_LAYERS", "S_TRAIN", "KD_TYPE", "KD_WEIGHT", "S_LOSS", "IN_BATCH_NEG", "HARD_NEG",
             "N_HARD", "POS_SAMPLING", "S_LR", "S_EPOCHS", "S_BATCH_SIZE", "S_WEIGHT_DECAY", "S_GRAD_CLIP", "SEED"]
_required += ["S_TAU"] if S_LOSS == "infonce" else ["S_MARGIN"] if S_LOSS == "margin" else []
_required += ["KD_TEMPERATURE"] if KD_TYPE == "listwise" else []
_required += ["S_LORMA_RANK", "S_LORMA_TARGETS"] if S_TRAIN == "lorma" else []
_required += ["HARD_TOP"] if HARD_NEG in ("bm25", "teacher", "model") and N_HARD else []
_unset = [n for n in _required if globals()[n] is None]
assert not _unset, f"fill in section [H] first: {_unset}"
assert S_TRAIN in ("full", "lorma") and KD_TYPE in ("listwise", "margin_mse") and S_LOSS in LOSSES
assert HARD_NEG in HARD_SOURCES + ("teacher",) and POS_SAMPLING in POS_SAMPLINGS
assert 0.0 <= KD_WEIGHT <= 1.0
assert IN_BATCH_NEG or (N_HARD and HARD_NEG != "none"), "no negatives at all: turn on IN_BATCH_NEG or hard negatives"
_nT = len(lm.core.encoder_layers)
assert KEEP_ENC_LAYERS and all(0 <= i < _nT for i in KEEP_ENC_LAYERS)
assert len(KEEP_ENC_LAYERS) < _nT, "the student must be smaller than the teacher (keep fewer encoder blocks)"


# -------------------------------------------------------------------------------------
# [ANCILLARY] run setup
# -------------------------------------------------------------------------------------
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
KD_NAME = f"c6_kd_pcr_{len(KEEP_ENC_LAYERS)}enc_{S_TRAIN}_{KD_TYPE}"
KD_DIR = f"/kaggle/working/{KD_NAME}"; os.makedirs(KD_DIR, exist_ok=True)
S_SAVE = f"{KD_DIR}/best_trainable.pt"
S_LAST = f"{KD_DIR}/last_checkpoint.pt"


# -------------------------------------------------------------------------------------
# [ANCILLARY] student backbone: fresh Phase 1 c-6, kept encoder blocks only, no decoder
# -------------------------------------------------------------------------------------
_ck = torch.load(f"{ROOT}/cs781_phase1_full/model_encdec/model_final.pt", map_location="cpu", weights_only=False)
lm_s = EncDecLM(EncDecConfig(**_ck["config"])); print("student load:", lm_s.load_state_dict(_ck["model"])); del _ck
lm_s.core.encoder_layers = nn.ModuleList([lm_s.core.encoder_layers[i] for i in KEEP_ENC_LAYERS])
lm_s.core.decoder_layers = nn.ModuleList()          # retrieval never uses the decoder
lm_s.to(device).eval()
print(f"student backbone: encoder blocks {list(KEEP_ENC_LAYERS)} of {_nT}, decoder removed")

# what trains: "full" = every student weight, "lorma" = frozen student + your LORMALinear (Cell K1)
for p in lm_s.parameters():
    p.requires_grad = (S_TRAIN == "full")
if S_TRAIN == "lorma":
    _ns = wrap_linears(lm_s, S_LORMA_TARGETS, lambda lin: LORMALinear(lin, S_LORMA_RANK).to(device))
    print(f"student: wrapped {_ns} linears with LORMA (I + BA)W, rank {S_LORMA_RANK}")
student = PCREncoder(lm_s, T_CFG["pool"]).to(device)
count_params(student)
S_PARAMS = retrieval_params(lm_s)
print(f"student / teacher retrieval parameters: {S_PARAMS / 1e6:.2f}M / {T_PARAMS / 1e6:.2f}M "
      f"({100 * S_PARAMS / T_PARAMS:.1f}%)")
assert S_PARAMS < T_PARAMS, "student is not smaller than the teacher"


# -------------------------------------------------------------------------------------
# [ANCILLARY] KD loss on the batch similarity matrices
# -------------------------------------------------------------------------------------
def kd_loss(S_s, S_t, allowed, target, kd_type, temperature):
    S_t = S_t.float().detach()
    if kd_type == "listwise":
        T = temperature
        log_ps = F.log_softmax((S_s / T).masked_fill(~allowed, -1e4), dim=-1)
        pt = F.softmax((S_t / T).masked_fill(~allowed, -1e4), dim=-1)
        kl = (pt * (torch.log(pt.clamp_min(1e-12)) - log_ps)).masked_fill(~allowed, 0.0).sum(-1)
        return kl.mean() * T * T
    B = S_s.size(0)
    neg = allowed.clone(); neg[torch.arange(B), target] = False
    m_s = S_s.gather(1, target[:, None]) - S_s                       # s(q, pos) - s(q, c)
    m_t = S_t.gather(1, target[:, None]) - S_t
    se = ((m_s - m_t) ** 2) * neg
    return (se.sum(1) / neg.sum(1).clamp_min(1)).mean()

# self-check: zero KD loss when the student equals the teacher
_S = torch.randn(3, 5); _al = torch.ones(3, 5, dtype=torch.bool); _tg = torch.tensor([0, 1, 2])
assert kd_loss(_S, _S, _al, _tg, "listwise", 2.0).abs() < 1e-5 and kd_loss(_S, _S, _al, _tg, "margin_mse", 1.0) < 1e-8
print("KD loss checks passed")

student load: <All keys matched successfully>
student backbone: encoder blocks [0, 2, 4, 6] of 8, decoder removed
trainable 16.548M / total 16.55M (100.00%)
student / teacher retrieval parameters: 16.55M / 30.92M (53.5%)
KD loss checks passed


## Cell K3 — KD training, teacher vs student, submission

In [11]:
# =====================================================================================
# CELL K3 — [ANCILLARY] train the student with KD, compare with the teacher, write the submission
# Run after K1 and K2. Best epoch by dev Micro-F1@7. Writes /kaggle/working/<KD_NAME>/submission.zip
# Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
GRAD_CHECKPOINT = GRAD_CHECKPOINT       # read by encoder_forward in E0

def train_kd(enc, lr, epochs, batch_size, weight_decay, grad_clip=None, lr_lambda=None, chunk=None,
             save_path="best_trainable.pt", last_path=None, resume_from=None, max_epoch_minutes=None):
    params = [p for p in enc.parameters() if p.requires_grad]
    assert params, "no trainable parameters"
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=_AUTOCAST)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda) if lr_lambda else None
    best, log, start = -1.0, [], 0
    if resume_from:
        ck = torch.load(resume_from, map_location="cpu", weights_only=False)
        enc.load_state_dict(ck["trainable"], strict=False)
        opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        if sched and ck.get("sched"):
            sched.load_state_dict(ck["sched"])
        best, log, start = ck["best"], ck["log"], ck["epoch"]
        torch.save(ck["best_trainable"], save_path)
        print(f"resumed after epoch {start} (best dev Micro-F1@7 {best:.4f})")
        del ck
    gold = GOLD["train"]
    hard_rank = None
    if HARD_NEG == "teacher":
        hard_rank = T_TRAIN_RANK
    elif HARD_NEG == "bm25" and N_HARD:
        if "train" not in BM25_RANK:
            BM25_RANK["train"] = bm25_rank("train", top=max(200, HARD_TOP))
        hard_rank = BM25_RANK["train"]
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    for ep in range(start, epochs):
        if HARD_NEG == "model" and N_HARD:
            hard_rank = rank_split(enc, "train", pieces["train_q"], pieces["train_c"], **RANKING,
                                   batch_windows=ENC_BATCH_WINDOWS, top=HARD_TOP + 50, q_ids=TRAIN_QIDS)
        enc.train()
        rows = make_epoch_rows(POS_SAMPLING, gold)
        batches = [rows[i:i + batch_size] for i in range(0, len(rows), batch_size)]
        t0, tot, tot_task, tot_kd, seen = time.time(), 0.0, 0.0, 0.0, 0
        pbar = tqdm(batches, desc=f"KD epoch {ep + 1}/{epochs}")
        for batch in pbar:
            negs = [pick_negatives(q, N_HARD, HARD_NEG, hard_rank, HARD_TOP, gold) for q, _ in batch]
            cand_ids = list(dict.fromkeys([p for _, p in batch] + [c for n in negs for c in n]))
            col = {c: j for j, c in enumerate(cand_ids)}
            own = [{col[c] for c in n} for n in negs]
            with torch.no_grad():
                S_t = teacher_scores(batch, cand_ids)
            parts = {}
            def loss_fn(qE, qM, cE, cM):
                S_s = similarity(qE, qM, cE, cM, **RANKING)
                allowed, target = batch_mask(batch, cand_ids, own, gold, IN_BATCH_NEG, S_s.device)
                task = score_loss(S_s, allowed, target, S_LOSS, S_TAU, S_MARGIN)
                kd = kd_loss(S_s, S_t.to(S_s.device), allowed, target, KD_TYPE, KD_TEMPERATURE)
                parts["task"], parts["kd"] = task.item(), kd.item()
                return (1 - KD_WEIGHT) * task + KD_WEIGHT * kd
            opt.zero_grad(set_to_none=True)
            l = step_backward(enc, [pieces["train_q"][q] for q, _ in batch],
                              [pieces["train_c"][c] for c in cand_ids], loss_fn, chunk, scaler)
            if grad_clip:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, grad_clip)
            scaler.step(opt); scaler.update()
            if sched:
                sched.step()
            tot += l; tot_task += parts["task"]; tot_kd += parts["kd"]; seen += 1
            pbar.set_postfix(loss=f"{tot / seen:.4f}", task=f"{tot_task / seen:.4f}", kd=f"{tot_kd / seen:.4f}")
            if max_epoch_minutes and time.time() - t0 > 60 * max_epoch_minutes:
                print(f"  time cap: stopped epoch after {seen}/{len(batches)} batches")
                break
        dev_rank = rank_split(enc, "dev", pieces["dev_q"], pieces["dev_c"], **RANKING, batch_windows=ENC_BATCH_WINDOWS)
        r = evaluate_ranking(dev_rank, GOLD["dev"], show=False)
        row = {"epoch": ep + 1, "train_loss": tot / max(1, seen), "task_loss": tot_task / max(1, seen),
               "kd_loss": tot_kd / max(1, seen), "batches": seen, "minutes": (time.time() - t0) / 60,
               **{k: r[k] for k in ("micro_f1", "micro_p", "micro_r", "recall@100", "mrr")}}
        log.append(row)
        improved = r["micro_f1"] > best
        if improved:
            best = r["micro_f1"]
            torch.save(trainable_state(enc), save_path)
        print(f"epoch {ep + 1}: loss {row['train_loss']:.4f} (task {row['task_loss']:.4f}, kd {row['kd_loss']:.4f}) | "
              f"dev Micro-F1@7 {r['micro_f1']:.4f} (P {r['micro_p']:.4f} R {r['micro_r']:.4f}) | "
              f"R@100 {r['recall@100']:.4f}{' *best*' if improved else ''} | {row['minutes']:.1f} min")
        if last_path:
            torch.save({"trainable": trainable_state(enc), "best_trainable": torch.load(save_path),
                        "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                        "sched": sched.state_dict() if sched else None, "epoch": ep + 1, "best": best,
                        "log": log}, last_path)
    return log, best


t_train = time.time()
S_log, S_best = train_kd(student, lr=S_LR, epochs=S_EPOCHS, batch_size=S_BATCH_SIZE, weight_decay=S_WEIGHT_DECAY,
                         grad_clip=S_GRAD_CLIP, lr_lambda=S_LR_LAMBDA, chunk=CHUNK, save_path=S_SAVE,
                         last_path=S_LAST, resume_from=S_RESUME_FROM, max_epoch_minutes=S_MAX_EPOCH_MINUTES)
S_train_min = (time.time() - t_train) / 60
S_train_mem = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0

student.load_state_dict(torch.load(S_SAVE), strict=False)
if torch.cuda.is_available():
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
_t0 = time.time()
S_dev_rank = rank_split(student, "dev", pieces["dev_q"], pieces["dev_c"], **RANKING,
                        batch_windows=ENC_BATCH_WINDOWS, top=100)
S_dev_s = time.time() - _t0
S_dev_mem = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
print("\n-- STUDENT on dev (best epoch)")
S_res = evaluate_ranking(S_dev_rank, GOLD["dev"], name="student dev")
breakdown(S_res)

# agreement with the teacher: overlap of the top-7 lists
_ov = np.mean([len(set(S_dev_rank[q][:TOPK]) & set(T_dev_rank[q][:TOPK])) / TOPK for q in T_dev_rank])
print("\n-- teacher vs student (dev)")
print(f"{'':10s} {'Micro-F1@7':>10s} {'P@7':>7s} {'R@7':>7s} {'R@100':>7s} {'params':>9s} {'dev rank s':>10s} {'mem GB':>7s}")
for name, r, n, s, m in (("teacher", T_res, T_PARAMS, T_dev_s, T_dev_mem), ("student", S_res, S_PARAMS, S_dev_s, S_dev_mem)):
    print(f"{name:10s} {r['micro_f1']:10.4f} {r['micro_p']:7.4f} {r['micro_r']:7.4f} {r['recall@100']:7.4f} "
          f"{n / 1e6:8.2f}M {s:10.0f} {m:7.2f}")
print(f"BM25 dev Micro-F1@7 {bm25_dev['micro_f1']:.4f} | student keeps {100 * S_res['micro_f1'] / max(1e-9, T_res['micro_f1']):.1f}% "
      f"of the teacher's score with {100 * S_PARAMS / T_PARAMS:.1f}% of its parameters | "
      f"top-7 overlap with teacher {_ov:.1%} | student training {S_train_min:.1f} min, peak {S_train_mem:.2f} GB")

json.dump({"run": KD_NAME, "teacher": T_CFG["run"], "keep_enc_layers": list(KEEP_ENC_LAYERS), "s_train": S_TRAIN,
           "kd_type": KD_TYPE, "kd_weight": KD_WEIGHT, "kd_temperature": KD_TEMPERATURE, "s_loss": S_LOSS,
           "hard_neg": HARD_NEG, "log": S_log, "train_minutes": S_train_min, "peak_mem_gb": S_train_mem,
           "teacher_params": T_PARAMS, "student_params": S_PARAMS, "top7_overlap": float(_ov),
           "teacher_dev": {k: T_res[k] for k in ("micro_f1", "micro_p", "micro_r", "recall@100", "mrr")},
           "student_dev": {k: S_res[k] for k in ("micro_f1", "micro_p", "micro_r", "recall@100", "mrr")},
           "teacher_dev_seconds": T_dev_s, "student_dev_seconds": S_dev_s},
          open(f"{KD_DIR}/kd_log.json", "w"), indent=1)

# Student test ranking -> /kaggle/working/<KD_NAME>/submission.zip
S_test_rank = rank_split(student, "test", pieces["test_q"], pieces["test_c"], **RANKING,
                         batch_windows=ENC_BATCH_WINDOWS, top=100)
write_prediction(S_test_rank, out_dir=KD_DIR)
print("KD outputs:", sorted(os.listdir(KD_DIR)))


/tmp/ipykernel_23/2609752769.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=_AUTOCAST)


BM25 index:   0%|          | 0/4320 [00:00<?, ?it/s]

BM25 train:   0%|          | 0/827 [00:00<?, ?it/s]

KD epoch 1/10:   0%|          | 0/205 [00:00<?, ?it/s]

encode dev pool:   0%|          | 0/1023 [00:00<?, ?it/s]

encode dev queries:   0%|          | 0/118 [00:00<?, ?it/s]

... (51 outputs left out)

encode test pool:   0%|          | 0/1727 [00:00<?, ?it/s]

encode test queries:   0%|          | 0/237 [00:00<?, ?it/s]

rank test:   0%|          | 0/1 [00:00<?, ?it/s]

wrote /kaggle/working/c6_kd_pcr_4enc_full_listwise/prediction.json (237 queries x 7) and /kaggle/working/c6_kd_pcr_4enc_full_listwise/submission.zip
KD outputs: ['best_trainable.pt', 'kd_log.json', 'last_checkpoint.pt', 'prediction.json', 'submission.zip']
